In [1]:
///- --package . --timeout 1200000
:< sm'_operators

:| test_status =
    | Passed
    | Failed
    | Static
    | Skipped
    | Unsupported
    | NotRun
    | TimedOut

:| reported_status =
    | Known : test_status
    | Unknown : string

:== test_row = { file : string; test : string; target : string; status : reported_status; ms : i32 }

:== file_row = { file : string; seconds : decimal::decimal; exit_code : i32 }

:== bench_row = { unit : string; parse_us : i32; write_us : i32; runs : i32; bytes : i32; os : string }

:| example_verdict =
    | Valid
    | Invalid : i32
    | NotWritten
    | Unmeasured

:== example_row = { example : string; bytes : string; verdict : example_verdict; first : string }

:= status_of (text : string) : reported_status =
    ?= case_labels::case_of_label "" "_" text ?|
    | Some s => Known s
    | None => Unknown text

:= status_text (s : reported_status) : string =
    ?= s ?|
    | Known known => case_labels::case_label "" "_" known
    | Unknown text => text

:= count_of (text : string) : i32 =
    ?= (convert::try_convert (sm'::trim text) : option i32) ?|
    | Some n => n
    | None => 0i32

:= seconds_of (text : string) : decimal::decimal =
    ?= decimal::parse (sm'::trim text) ?|
    | Some d => d
    | None => decimal::of_int 0i64

:= written_of (text : string) : bool =
    ?= (convert::try_convert (sm'::trim text) : option i32) ?|
    | Some n => n <> 0i32
    | None => !+

:= verdict_of (written : bool) (messages : string) : example_verdict =
    ? written = !- ?> NotWritten
    |? sm'::trim messages = "" ?> Unmeasured
    |? count_of messages = 0i32 ?> Valid
    |: Invalid (count_of messages)

:= tsv_rows (text : string) : list (list string) =
    ?= sm'::split_list "\n" (sm'::replace "\r" "" text) ?|
    | _ :: rows => listm'::filter (\ (r : list string) => r <> [ "" ]) (listm::map (sm'::split_list "\t") rows)
    | [] => []

:= cell (columns : list string) (at : i32) : string =
    ?= listm'::try_item at columns ?|
    | Some text => text
    | None => ""

:= test_row_of (c : list string) : test_row =
    {
        file = cell c 0i32
        test = cell c 1i32
        target = cell c 2i32
        status = status_of (cell c 3i32)
        ms = count_of (cell c 4i32)
    }

:= file_row_of (c : list string) : file_row =
    { file = cell c 0i32; seconds = seconds_of (cell c 1i32); exit_code = count_of (cell c 2i32) }

:= bench_row_of (c : list string) : bench_row =
    {
        unit = cell c 0i32; parse_us = count_of (cell c 1i32); write_us = count_of (cell c 2i32)
        runs = count_of (cell c 3i32); bytes = count_of (cell c 4i32); os = cell c 5i32
    }

:= example_row_of (c : list string) : example_row =
    {
        example = cell c 0i32
        bytes = cell c 1i32
        verdict = verdict_of (written_of (cell c 5i32)) (cell c 2i32)
        first = cell c 4i32
    }

:= escaped (text : string) : string = sm'::replace "|" "\\|" (sm'::replace "\n" " " text)

:= table_row (cells : list string) : string = "| " ++# sm'::concat_list " | " (listm::map escaped cells) ++# " |"

:= table (header : list string) (rows : list (list string)) : list string =
    table_row header
    :: ("|" ++# sm'::concat_list "|" (listm::map (\ _ => "---") header) ++# "|")
    :: listm::map table_row rows

:~ ^ slower_first (row : file_row) (sorted : list file_row) : list file_row =
    ?= sorted ?|
    | head :: rest =>
        ? decimal::compare row::seconds head::seconds >= 0i32 ?> row :: sorted |: head :: slower_first row rest
    | [] => [ row ]

:= by_seconds (rows : list file_row) : list file_row = listm::fold (\ sorted row => slower_first row sorted) [] rows

:= verdict_text (v : example_verdict) : string =
    ?= v ?|
    | Invalid n => sm'::obj_to_string n ++# " messages"
    | _ => case_labels::label_of_name "" " " (reflection::case_name v)

:= has_status (s : test_status) (r : test_row) : bool = r::status = Known s

:= is_unknown (r : test_row) : bool =
    ?= r::status ?|
    | Unknown _ => !+
    | Known _ => !-

:= counted (rows : list test_row) (s : test_status) : i32 = listm::length (listm'::filter (has_status s) rows)

:= other_count (rows : list test_row) : i32 =
    counted rows NotRun
    + counted rows Unsupported
    + counted rows TimedOut
    + listm::length (listm'::filter is_unknown rows)

:= count_part (n : i32) (label : string) : string = sm'::obj_to_string n ++# " " ++# label

:= main_counts (rows : list test_row) : list string =
    [ count_part (counted rows Passed) "passed"; count_part (counted rows Failed) "failed" ]

:= status_counts (rows : list test_row) : string =
    := rest = [ count_part (counted rows Static) "static"; count_part (counted rows Skipped) "skipped" ]
    sm'::concat_list ", " (main_counts rows ++ rest ++ [ count_part (other_count rows) "other" ])

:= test_cells (r : test_row) : list string =
    [ r::file; r::test; r::target; status_text r::status; sm'::obj_to_string r::ms ]

:= file_cells (r : file_row) : list string = [ r::file; decimal::text r::seconds; sm'::obj_to_string r::exit_code ]

:= bench_cells (r : bench_row) : list string =
    [
        r::unit; sm'::obj_to_string r::parse_us; sm'::obj_to_string r::write_us; sm'::obj_to_string r::runs
        sm'::obj_to_string r::bytes; r::os
    ]

:= example_cells (r : example_row) : list string = [ r::example; r::bytes; verdict_text r::verdict; r::first ]

:= passing (r : test_row) : bool = has_status Passed r || has_status Static r

:= run_link (run : string) : string = "[" ++# run ++# "](https://github.com/i574n/fiscal/actions/runs/" ++# run ++# ")"

:= local_stamp (built : string) : string = "local build" ++# (? built = "" ?> "" |: ", " ++# built)

:= ci_stamp (sha : string) (built : string) : string =
    := run = run_link (env::get_environment_variable "GITHUB_RUN_ID")
    "commit `" ++# sha ++# "`, run " ++# run ++# ", built " ++# built

:= stamp () : string =
    := sha = env::get_environment_variable "GITHUB_SHA"
    := built = env::get_environment_variable "FISCAL_BUILD_TIME"
    ? sha = "" ?> local_stamp built |: ci_stamp sha built

:= section (title : string) (body : list string) : list string = [ ""; "## " ++# title; "" ] ++ body

:= failing_lines (tests : list test_row) : list string =
    := not_passed = listm'::filter (\ (r : test_row) => passing r = !-) tests
    ? not_passed = [] ?> [ "Every test passed." ]
    |: table [ "file"; "test"; "target"; "status"; "ms" ] (listm::map test_cells not_passed)

:= tests_section (tests : list test_row) : list string =
    section "Tests" ((status_counts tests ++# ".") :: "" :: failing_lines tests)

:= files_section (files : list file_row) : list string =
    section "Slowest files" (table [ "file"; "seconds"; "exit" ] (listm::map file_cells (listm'::take_ 10i32 (
        by_seconds files))))

:= bench_section (bench : list bench_row) : list string =
    section
        "Bench (median microseconds)"
        (table [ "unit"; "parse µs"; "write µs"; "runs"; "bytes"; "os" ] (listm::map bench_cells bench))

:= examples_section (examples : list example_row) : list string =
    section "Examples" (table [ "example"; "bytes"; "verdict"; "first message" ] (listm::map example_cells examples))

:= is_derived (g : canon_coverage::gap) : bool = g = canon_coverage::Derived

:= is_not_carried (g : canon_coverage::gap) : bool = g = canon_coverage::NotCarried

:= is_national (g : canon_coverage::gap) : bool = is_derived g = !- && is_not_carried g = !-

:= gap_count (wanted : canon_coverage::gap -> bool) (gaps : list (canon::term_id * canon_coverage::gap)) : string =
    sm'::obj_to_string (listm::length (listm'::filter (\ (_, g) => wanted g) gaps) : i32)

:== census =
    {
        family : string
        bindings : list (canon::term_id * string)
        implied : list (canon::term_id * string)
        gaps : list (canon::term_id * canon_coverage::gap)
    }

:= gap_kinds (gaps : list (canon::term_id * canon_coverage::gap)) : list string =
    [ gap_count is_derived gaps; gap_count is_national gaps; gap_count is_not_carried gaps ]

:= required_only \* t.
    (required : list canon::term_id) (pairs : list (canon::term_id * t))
    : list (canon::term_id * t)
    =
    listm'::filter (\ (id, _) => listm'::contains id required) pairs

:= census_counts (required : list canon::term_id) (c : census) : list i32 =
    [
        listm::length (listm'::filter (canon_coverage::is_bound c::bindings) required)
        listm::length (required_only required c::implied)
    ]

:= census_cells (required : list canon::term_id) (c : census) : list string =
    := counts = listm::map sm'::obj_to_string (census_counts required c)
    c::family :: listm::append counts (gap_kinds (required_only required c::gaps))

:= nfe_census () : census =
    {
        family = "NF-e 4.00"
        bindings = canon_nfe::nfe_bindings ()
        implied = canon_nfe::nfe_implied ()
        gaps = canon_nfe::nfe_gaps ()
    }

:= cfdi_census () : census =
    {
        family = "CFDI 4.0"
        bindings = canon_cfdi::cfdi_bindings ()
        implied = canon_cfdi::cfdi_implied ()
        gaps = canon_cfdi::cfdi_gaps ()
    }

:= census_rows () : list (list string) =
    listm::map (census_cells (canon_coverage::required_ids (canon::en16931_terms ()))) [ nfe_census (); cfdi_census () ]

:= census_section () : list string =
    := header = [ "family"; "bound"; "implied"; "derived"; "national code"; "not carried" ]
    section "Canon census (the 24 required EN 16931 terms)" (table header (census_rows ()))

:= rendered
    (tests : list test_row) (files : list file_row) (bench : list bench_row) (examples : list example_row)
    : string
    =
    := body =
        tests_section tests
        ++ files_section files
        ++ bench_section bench
        ++ examples_section examples
        ++ census_section ()
    sm'::concat_list "\n" ([ "# fiscal CI reports"; ""; stamp () ] ++ body ++ [ "" ])

:= metrics_rows (kind : string) : list (list string) =
    := path =
        fiscal_paths::under (
            "target/ci-metrics/" ++# kind ++# env::get_environment_variable "FISCAL_METRICS_SUFFIX" ++# ".tsv"
        )
    ? xml::is_file path ?> tsv_rows (file_system::read_all_text path) |: []

unsupported on this machine: `zig` was not found

#### tsv_rows_read_into_their_typed_columns (test)

In [2]:
///- --test
///> rust

:= rows =
    tsv_rows "File\tTest\tTarget\tStatus\tMs\r\ngenerator\ta\trust\tpassed\t12\r\ncanon\tb\trust\tnot_run\tx\r\n"
:= typed = listm::map test_row_of rows
? listm::map (\ (r : test_row) => r::status) typed <> [ Known Passed; Known NotRun ]
?> failwith "statuses read as cases"
? listm::map (\ (r : test_row) => r::ms) typed <> [ 12i32; 0i32 ] ?> failwith "ms reads as i32, junk as 0"
? status_text (status_of "flaky") <> "flaky" ?> failwith "an unknown status keeps its text"
? verdict_of !+ "3" <> Invalid 3i32 || verdict_of !- "" <> NotWritten ?> failwith "verdicts read"
? listm::map verdict_text [ Valid; Invalid 2i32; NotWritten; Unmeasured ]
    <> [ "valid"; "2 messages"; "not written"; "unmeasured" ]
?> failwith "each verdict prints as its case label"
? written_of "0" || written_of "1" = !- || written_of "" = !-
?> failwith "Written reads 0 as not written, 1 or a missing column as written"

0

#### report_tables_escape_and_rank (test)

In [3]:
///- --test
///> rust

? table_row [ "a|b"; "c" ] <> "| a\\|b | c |" ?> failwith "a pipe inside a cell is escaped"
:= files = listm::map file_row_of [ [ "x"; "1.5"; "0" ]; [ "y"; "12.0"; "1" ]; [ "z"; "3"; "0" ] ]
? listm::map (\ (r : file_row) => r::file) (by_seconds files) <> [ "y"; "z"; "x" ] ?> failwith "slowest file first"

0

#### status_counts_cover_every_status (test)

In [4]:
///- --test
///> rust

:= row (status : string) : test_row = test_row_of [ "f"; "t"; "rust"; status; "1" ]
:= rows = listm::map row [ "passed"; "passed"; "failed"; "static"; "skipped"; "not_run"; "weird" ]
? status_counts rows <> "2 passed, 1 failed, 1 static, 1 skipped, 2 other"
?> failwith "the summary counts each status, with not_run and unknown ones as other"

0

#### writes_the_ci_reports (test)

In [5]:
///- --test
///> rust

:= text =
    rendered
        (listm::map test_row_of (metrics_rows "tests")) (listm::map file_row_of (metrics_rows "files"))
        (listm::map bench_row_of (metrics_rows "bench")) (listm::map example_row_of (metrics_rows "examples"))
file_system::directory_create_all (fiscal_paths::under "target/ci-metrics") |> ignore
file_system::write_all_text (fiscal_paths::under "target/ci-metrics/reports.md") text
? sm'::starts_with "# fiscal CI reports" text = !- ?> failwith "the report has its title"

0

#### the_canon_census_counts_each_family (test)

In [6]:
///- --test
///> rust

? census_rows () <> [ [ "NF-e 4.00"; "11"; "2"; "4"; "4"; "3" ]; [ "CFDI 4.0"; "11"; "1"; "5"; "3"; "4" ] ]
?> failwith "each family's 24 required terms split into bound, implied and the three gap kinds"

0